# 정규표현식 사용

In [16]:
import re
text = "주문번호: 12345 금액: 89,000원 이메일: test@test.com"
emails = re.findall(r"\w+@\w+\.\w+", text)
numbers = re.findall(r"\d+", text)
price = re.findall(r"\d{1,3},\d{3}", text)
print('Emails:', emails)
print('Numbers:', numbers)
print('Price:', price)

Emails: ['test@test.com']
Numbers: ['12345', '89', '000']
Price: ['89,000']


# IMAP 메일 가져오기

In [17]:
import imaplib
mail = imaplib.IMAP4_SSL('imap.gmail.com')
mail.login('mhnkms8041@gmail.com','qxvh ihup erwd ppgn')
mail.select('inbox')
print('Connected to mailbox')

Connected to mailbox


# 메일 본문 데이터 처리

In [18]:
body = "주문번호: 98765 금액: 120,000원"
order_id = re.findall(r"\d{5}", body)
price = re.findall(r"\d{1,3},\d{3}", body)
print(order_id, price)

['98765'] ['120,000']


# CSV 저장 

In [19]:
import csv
data = [['order_id', 'price'],['98765', '120000']]
with open('result.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(data)
print('CSV 저장 완료')

CSV 저장 완료


# 메일에서 첨부파일 내려받아 텍스트 추출하기

In [20]:
import imaplib
import email
from email.header import decode_header
import os
import pdfplumber
IMAP_SERVER = "imap.gmail.com"
EMAIL = 'mhnkms8041@gmail.com'
PASSWORD = 'qxvh ihup erwd ppgn'
SAVE_DIR = "./downloads"
os.makedirs(SAVE_DIR, exist_ok=True)
# 1. IMAP 접속
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("inbox")

('OK', [b'15'])

In [21]:
# 2. 메일 검색 (제목 필터는 직접 처리)
result, data = mail.search(None, "ALL")
mail_ids = data[0].split()
mail_csv_list = []
target_mail = None
# 최근 메일부터 역순 탐색
for mail_id in reversed(mail_ids):
    result, data = mail.fetch(mail_id, "(RFC822)")
    raw_email = data[0][1]
    msg = email.message_from_bytes(raw_email)
    # 제목 디코딩
    subject, encoding = decode_header(msg["Subject"])[0]
    if isinstance(subject, bytes):
        subject = subject.decode(encoding or "utf-8")
    if "[업무협조]" in subject:
        print("대상 메일:", subject)
        target_mail = msg
        mail_csv_list
        # break
    
if target_mail is None:
    print(" 조건에 맞는 메일 없음")
# exit()

대상 메일: [업무협조] 데이터 검증 요청
대상 메일: Fwd: [업무협조] !! 기습메일 !! 중요한 내용입니다 필독바람
대상 메일: [업무협조] !! 기습메일 !! 중요한 내용입니다 필독바람


In [22]:
# 3. PDF 첨부파일 다운로드
pdf_files = []
for part in target_mail.walk():
    content_disposition = part.get("Content-Disposition")
    if content_disposition and "attachment" in content_disposition:
        filename = part.get_filename()

        if filename:
            filename, enc = decode_header(filename)[0]
            if isinstance(filename, bytes):
                filename = filename.decode(enc or "utf-8")
            if filename.lower().endswith(".pdf"):
                filepath = os.path.join(SAVE_DIR, filename)
                with open(filepath, "wb") as f:
                    f.write(part.get_payload(decode=True))
                print(f"다운로드 완료: {filepath}")
                pdf_files.append(filepath)

In [23]:
# 4. PDF 텍스트 추출
for pdf_path in pdf_files:
    print(f"\n텍스트 추출: {pdf_path}")
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
print("---- 추출 텍스트 ----")
print(full_text[:1000]) # 일부만 출력

---- 추출 텍스트 ----



# 메일 -> CSV

In [47]:
# 메일 분류
def classify_mail(subject, body):
    text = (subject + " " + body).lower()

    categories = {
        "업무협조": [
            "협조", "요청", "부탁", "전달", "회신", "확인", "처리", "검토", "승인", "진행", "지원", "일청 요청", "자료 요청", "공유 요청", "확인 부탁", "전달 부탁"
        ],
        "보고서": [
            "보고", "결과", "분석", "현황", "매출", "데이터", "통계", "성과", "개선", "지표", "리포트", "요약"
        ],
        "회의록": [
            "회의", "회의록", "안건", "참석자", "일정", "논의", "결정", "진행", "의결", "회의 결과"
        ],
        "공지": [
            "공지", "안내", "알림", "변경", "일정", "공지사항", "배포", "적용", "공지드립니다", "참고"
        ]
    }

    scores = {key: 0 for key in categories}

    for category, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                scores[category] += 1

    # 점수 가장 높은 카테고리 선택
    best_category = max(scores, key=scores.get)

    # 전부 0이면 fallback
    if scores[best_category] == 0:
        return "공지"  # 기본값 (가장 범용적)

    return best_category

In [48]:
# 액션 분류
def detect_action(body):
    need_reply_keywords = [
        "회신", "답변", "의견 부탁", "피드백", "확인 부탁",
        "검토 후 회신", "회신 요청", "답장"
    ]

    # no_reply_keywords = [
    #     "참고", "공유드립니다", "공지", "안내", "전달드립니다"
    # ]

    for kw in need_reply_keywords:
        if kw in body:
            return "회신 필요"

    # for kw in no_reply_keywords:
    #     if kw in body:
    #         return "회신 불필요"

    return "회신 불필요"

In [49]:
# 기한 추출
def extract_deadline(body):
    patterns = [
        r"\d{4}[.-]\d{2}[.-]\d{2}",
        r"\d{2}[.-]\d{2}",
        r"\d{1,2}월\s?\d{1,2}일",
        r"\d{1,2}/\d{1,2}"
    ]

    for pattern in patterns:
        match = re.search(pattern, body)
        if match:
            return match.group()

    # "까지", "마감" 같은 문맥 기반 탐색
    deadline_keywords = ["까지", "마감", "기한"]
    for line in body.split("\n"):
        if any(k in line for k in deadline_keywords):
            return line.strip()

    return "-"

In [50]:
import imaplib
import email
from email.header import decode_header
import os
import pdfplumber
IMAP_SERVER = "imap.gmail.com"
EMAIL = 'mhnkms8041@gmail.com'
PASSWORD = 'qxvh ihup erwd ppgn'
SAVE_DIR = "./downloads"
os.makedirs(SAVE_DIR, exist_ok=True)
# 1. IMAP 접속
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("inbox")

('OK', [b'15'])

In [51]:
# 2. 메일 검색 (제목 필터는 직접 처리)
result, data = mail.search(None, "ALL")
mail_ids = data[0].split()
mail_csv_list = []
target_mail = None
# 최근 메일부터 역순 탐색
for mail_id in reversed(mail_ids):
    result, data = mail.fetch(mail_id, "(RFC822)")
    raw_email = data[0][1]
    msg = email.message_from_bytes(raw_email)

    subject, encoding = decode_header(msg["Subject"])[0]
    if isinstance(subject, bytes):
        subject = subject.decode(encoding or "utf-8")

    # 필터 없이 전부 저장하거나 조건 유지 가능
    if "[업무협조]" in subject:
        print("대상 메일:", subject)
        mail_csv_list.append(msg)

대상 메일: [업무협조] 데이터 검증 요청
대상 메일: Fwd: [업무협조] !! 기습메일 !! 중요한 내용입니다 필독바람
대상 메일: [업무협조] !! 기습메일 !! 중요한 내용입니다 필독바람


In [52]:
import csv
import re

csv_path = "email_list.csv"
file_exists = os.path.isfile(csv_path)

with open(csv_path, "a", newline="", encoding="utf-8-sig") as csvfile:
    writer = csv.writer(csvfile)

    if not file_exists:
        # writer.writerow(["제목", "발신자", "분류", "액션", "기한", "요약"])
        writer.writerow(["제목", "발신자", "분류", "액션", "기한"])

    for mail_item in mail_csv_list:

        subject, encoding = decode_header(mail_item["Subject"])[0]
        if isinstance(subject, bytes):
            subject = subject.decode(encoding or "utf-8")

        from_ = email.header.make_header(decode_header(mail_item.get("From")))

        # 메일별 PDF 처리
        full_text = ""
        for part in mail_item.walk():
            content_disposition = part.get("Content-Disposition")

            if content_disposition and "attachment" in content_disposition:
                filename = part.get_filename()

                if filename:
                    filename, enc = decode_header(filename)[0]
                    if isinstance(filename, bytes):
                        filename = filename.decode(enc or "utf-8")

                    if filename.lower().endswith(".pdf"):
                        filepath = os.path.join(SAVE_DIR, filename)

                        with open(filepath, "wb") as f:
                            f.write(part.get_payload(decode=True))

                        print(f"다운로드 완료: {filepath}")

                        # PDF 텍스트 추출
                        with pdfplumber.open(filepath) as pdf:
                            for page in pdf.pages:
                                text = page.extract_text()
                                if text:
                                    full_text += text + "\n"

        # 분석
        category = classify_mail(subject, full_text)
        action = detect_action(full_text)
        deadline = extract_deadline(full_text)
        # summary = summarize_text(full_text)

        # CSV 저장
        writer.writerow([
            subject,
            from_,
            category,
            action,
            deadline,
            # summary
        ])

print(f"\nCSV 저장 완료: {csv_path}")

다운로드 완료: ./downloads\mail1_request.pdf
다운로드 완료: ./downloads\mail2_request.pdf
다운로드 완료: ./downloads\mail1_request.pdf

CSV 저장 완료: email_list.csv
